# Fetch weather data from Copernicus

This file fetches weather data from Copernicus and 

In [ ]:
"""
sweden_energy_atlite.py
-----------------------

Hourly heating- and cooling-degree demand per Swedish county
using **atlite** + ERA5.

Prerequisites
-------------
conda create -n sweden_energy python=3.11
conda activate sweden_energy
pip install "atlite>=0.2.6" cdsapi geopandas rioxarray xarray dask[complete] \
            pyproj regionmask netcdf4 tqdm

Make sure your CDS API key is in ~/.cdsapirc  (see https://cds.climate.copernicus.eu/api-how-to)

Inputs
------
* sweden_counties.geojson  – county polygons (EPSG:4326 or any CRS)

Outputs
-------
* county_hourly_degree_hours.nc  –  DataArray(time, county, {HDD,CDD})
"""

import atlite
import geopandas as gpd
import xarray as xr
from pathlib import Path
from datetime import date
import numpy as np

# --------------------------------------------------------------------------------------
# 1. USER CONFIG
# --------------------------------------------------------------------------------------
START, END = date(2020, 1, 1), date(2024, 12, 31)  # inclusive
ERA_RES     = 0.25                                  # °  (ERA5 grid)
T_HEAT, T_COOL = 17.0, 22.0                         # °C HDD / CDD base
COUNTY_GEOJSON = "sweden_counties.geojson"
CUTOUT_DIR     = "atlite_cutouts"
OUTFILE        = "county_hourly_degree_hours.nc"
# --------------------------------------------------------------------------------------

cut_name = f"se_{START.year}_{END.year}_era5"
cut_path = Path(CUTOUT_DIR) / cut_name

print(">>> 1. Build / load cutout …")
cutout = atlite.Cutout(
    path=cut_path,
    module="era5",
    x=slice(10.5, 24.5),         # lon W–E
    y=slice(55.0, 69.5),         # lat S–N
    time=f"{START}/{END}",
    dx=ERA_RES, dy=ERA_RES,
).prepare()                      # downloads ERA5 once, reuses next time

print(">>> 2. County ↦ layout mask …")
counties = gpd.read_file(COUNTY_GEOJSON).to_crs(4326)
layout   = cutout.regionmask(counties, name="county")   # (region, y, x) weights

print(">>> 3. Hourly HDD / CDD …")
# atlite >=0.2.6 exposes heat_demand / cooling_demand returning degree-hours
hdd = cutout.heat_demand(layout, temp_threshold=T_HEAT)      # (time, county)
cdd = cutout.cooling_demand(layout, temp_threshold=T_COOL)

deg_hours = xr.merge([hdd.rename("HDD"), cdd.rename("CDD")]) \
              .assign_coords(county=("county", counties["name"]))

print(">>> 4. Save NetCDF …")
deg_hours.to_netcdf(OUTFILE, encoding={v: {"zlib": True, "complevel": 4}
                                       for v in deg_hours.data_vars})

print(f"Done ✅  ->  {OUTFILE}")
